In [ ]:
import sys
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/")
sys.path.append("/Users/ronguy/Dropbox/Work/CyTOF/Code/")
%load_ext autoreload
%autoreload 2
from CyTOFHelper import *
from TestHet import *
from PermCell_Smooth import *
#from SHAPset import *
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
Run="Corrs"
import xgboost as xgb
import umap
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import pandas as pd
from POgSET import *
import icecream as ic

In [ ]:
%matplotlib inline

In [ ]:
import glob

In [ ]:
metals_names_map = {' In115Di':'H3', ' Ce140Di':'Cytokeratin5', ' Nd142Di':'H3K27me2', ' Nd143Di':'p53', ' Nd144Di':'EZH2', ' Nd145Di':'H3K4me3',
                   ' Sm149Di':'H3K36me2', ' Nd150Di':'H3K4me1', ' Eu151Di':'H3K9me2', ' Sm152Di':'H4K16ac', ' Eu153Di':'H2AK119Ub', ' Gd155Di':'H3.3',
                   ' Gd156Di':'H3K64ac', ' Gd158Di':'ZEB1', ' Tb159Di':'H4', ' Gd160Di':'H3K27ac', ' Dy161Di':'H4K20me3', ' Ho165Di':'H3K36me3',
                   ' Er168Di':'H3K27me3', ' Tm169Di':'H3K9ac', ' Er170Di':'H3K9me3', ' Lu175Di':'H3S28p', ' Pr141Di':'human-EpCAM',
                    ' Sm147Di':'yH2A.X', ' Sm154Di':'Vimentin', ' Dy163Di':'ER', ' Dy164Di':'CD49f', ' Er166Di':'CD24', ' Er167Di':'GATA3',
                   ' Yb171Di':'CD44', ' Yb172Di':'Ki-67', ' Yb174Di':'K8_18'}

In [ ]:
R1=dict([(f[0][1:],f[1]) for f in metals_names_map.items()])

In [ ]:
R1

In [ ]:
dir="/Users/ronguy/Dropbox/CyTOF_Breast/data_revision_CR7_Nature/CyTOF1_BCcell-lines/csv_files/"
MCF7=pd.read_csv(dir+"c18_export_CyTOF_BClines_christi_06Feb2023_01_1_0_Time, Width subset_MCF7_Yael.csv")
#MCF7=pd.read_csv(dir+"c17_export_CyTOF_BClines_christi_06Feb2023_01_1_0_Time, Width subset_MCF7_Ori.csv")


In [ ]:
MCF7.columns=[f.strip(" ") for f in MCF7.columns]

In [ ]:
MCF7.columns

In [ ]:
MCF7.rename(columns=R1,inplace=True)

In [ ]:
MCF7

In [ ]:
list(MCF7.columns)

In [ ]:
N=[
 'Cytokeratin5',
 'H4K20me3',
 
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'H3K27me3',
 'H3K9me3',
 'H3K9me2',
 'H2AK119Ub',
 'H3.3',
 'H3K64ac',
 'ZEB1',
 'H3K27ac',
 'H3K36me3',
 'H3',
 'H3S28p',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'H3K4me1',
 'human-EpCAM',
 'yH2A.X',
 'H3K36me2',
 'H4K16ac',
 'Vimentin',
 'H4',
 'H3K9ac',
 'CD44',
 'Ki-67',
 'K8_18',
]

In [ ]:
#dir="/Users/ronguy/Dropbox/WIS-CIMA colab - Analysis/#3 CyTOF  - KPC sample, after CD45 depletion/"

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)

sns.set_style("white")


In [ ]:
Rep=dict(pd.read_excel("/Users/ronguy/Dropbox/Work/CyTOF/Mapping.xlsx").iloc[:,:].values)

In [ ]:
MCF7=MCF7[N]

In [ ]:
Rep['human-EpCAM']='EpCAM'


In [ ]:
DBs=['MCF7']

In [ ]:
MCF7.columns

In [ ]:
for DB in DBs:
#    globals()[DB]=pd.read_csv(F)
    globals()[DB].rename(columns=Rep,inplace=True)
    try:
        globals()[DB].drop(columns=['DNA1','DNA2','Event #'],inplace=True)        
    except:
        pass




In [ ]:
N=list(globals()[DBs[0]].columns)
N.sort()
#N.remove('N-cadherin')
NamesAll,EpiCols,NormMRK,CellIden,CellCyle=GetMarkers(N)


In [ ]:
NamesAll

In [ ]:
for DB in DBs:
    globals()[DB]=globals()[DB][NamesAll]
#    globals()[DB]['Samp']=DB

In [ ]:
GateColumns=['H3.3','H4','H3']#,'H3']#,'H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data

MCF7=Gate(MCF7,"MCF7")

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']+d.std()['H4']+d.std()['H3']

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[innerCols]=data[innerCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.3,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf, ddf,Q,M,M1,M2),method='cg')
    AA=out.params['a'].value

    M=M1*AA+M2*(1-AA)
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[innerCols]=data[innerCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:
from lmfit import minimize, Parameters

In [ ]:
innerCols=NormMRK

In [ ]:
MCF7=NormalizeNew(MCF7)

In [ ]:
scFac=5
MCF7=np.arcsinh(MCF7/scFac)

In [ ]:
NC=2000
aaaa=pd.DataFrame(columns=globals()[DBs[0]].columns)
for DB in DBs:
    aaaa=pd.concat([aaaa,globals()[DB].sample(NC,replace=False)]).copy()
                  

                
m=np.mean(aaaa,axis=0)
s=np.std(aaaa,axis=0)

for DB in DBs:
    globals()[DB]=(globals()[DB]-m)/s
    globals()[DB]['Samp']=DB


In [ ]:
%matplotlib inline
hKWD={'element':'step','fill':False,'stat':'density'}

In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3')
EPC.remove('H3.3')
EPC.remove('H4')
CNum=60000

In [ ]:
UM=umap.UMAP(min_dist=0.001,n_neighbors=40,random_state=42,verbose=True)

In [ ]:
MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')

In [ ]:
X_2d=UM.fit_transform(MCF7[NamesAll],ncols=5)

In [ ]:
plt.scatter(X_2d[:,0],X_2d[:,1])

In [ ]:

def ManualSelection(df, id_column="region_id", x_col="x", y_col="y"):
    df = df.copy()
    if id_column not in df.columns:
        df[id_column] = -1
    label_column = f"{id_column}_Label"
    if label_column not in df.columns:
        df[label_column] = ""

    df["idx"] = df.index
    selection_counter = {"count": 0}
    selected_indices = set()
    labels_dict = {}

    color_list = ['lightgray'] + list(plt.cm.tab10.colors)
    cmap = ListedColormap(color_list)
    norm = BoundaryNorm(boundaries=np.arange(-1.5, len(color_list) - 0.5), ncolors=len(color_list))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 6))
    fig.canvas.header_visible = False
    fig.canvas.footer_visible = False
    fig.canvas.toolbar_visible = True
    fig.canvas.resizable = True

    # Scatter plots
    sc1 = ax1.scatter(df[x_col], df[y_col], s=8, c=[0]*len(df), cmap=cmap, norm=norm)
    ax1.set_title("Manual Selection (Colored by Group)")

    valid_dropdown_columns = [col for col in df.columns if col not in [x_col, y_col, id_column, label_column]]
    default_dropdown_value = valid_dropdown_columns[0] if valid_dropdown_columns else None
    vmin, vmax = np.quantile(df[default_dropdown_value], 0.01), np.quantile(df[default_dropdown_value], 0.99)
    scatter2 = ax2.scatter(df[x_col], df[y_col], s=8, vmin=vmin, vmax=vmax,
                           c=df[default_dropdown_value] if default_dropdown_value else [0]*len(df),
                           cmap='seismic')
    ax2.set_title(f"Colored by: {default_dropdown_value}" if default_dropdown_value else "No column available")

    # Widgets
    output = widgets.Output()
    commit_button = widgets.Button(description="Commit Selection")
    finish_button = widgets.Button(description="Finish")
    label_text = widgets.Text(description="Label:", placeholder="Enter label for selection")
    label_text.layout.display = "none"
    control_box = widgets.HBox([commit_button, finish_button])
    legend_ref = {"widget": widgets.HTML()}
    dropdown = widgets.Dropdown(options=valid_dropdown_columns, value=default_dropdown_value)
    vbox = widgets.VBox([control_box, label_text, dropdown, legend_ref["widget"]])
    display(vbox, output)

    # Selection state
    data_pts = df[[x_col, y_col]].values
    highlight_plot = {"plot": None}
    highlight_plot2 = {"plot": None}
    live_poly1 = {"patch": None}
    live_poly2 = {"patch": None}
    selector_ref = {"selector": None}

    def refresh_plot():
        colors = df[id_column].map(lambda v: v if v >= 0 else -1).values
        sc1.set_array(colors)
        fig.canvas.draw_idle()

        # Legend
        legend_items = []
        for k, v in labels_dict.items():
            rgba = mcolors.to_rgba(color_list[(k % (len(color_list)-1)) + 1])
            hex_color = mcolors.to_hex(rgba)
            legend_items.append(
                f'<div style="margin:4px 0;">'
                f'<span style="display:inline-block;width:12px;height:12px;'
                f'background-color:{hex_color};border:1px solid #444;margin-right:6px;"></span>'
                f'{v}</div>'
            )
        legend_html = "<div><b>Legend</b>" + "".join(legend_items) + "</div>"
        legend_ref["widget"].value = f'<div style="font-family:sans-serif;font-size:13px;line-height:1.6;">{legend_html}</div>'

    class DualLasso:
        def __init__(self, ax1, ax2, onselect):
            self.ax1 = ax1
            self.ax2 = ax2
            self.onselect = onselect
            self.verts = []

            self.poly1 = Polygon(np.empty((0, 2)), closed=True, edgecolor='black', facecolor='none', linestyle='--', linewidth=1)
            self.poly2 = Polygon(np.empty((0, 2)), closed=True, edgecolor='yellow', facecolor='none', linestyle='--', linewidth=1)

            self.ax1.add_patch(self.poly1)
            self.ax2.add_patch(self.poly2)

            self.lasso = LassoSelector(ax1, onselect=self._on_select, useblit=True)
            self.cid_motion = fig.canvas.mpl_connect("motion_notify_event", self._on_move)
            self.cid_press = fig.canvas.mpl_connect("button_press_event", self._on_press)
            self.cid_release = fig.canvas.mpl_connect("button_release_event", self._on_release)

        def _on_press(self, event):
            self.verts = []

        def _on_move(self, event):
            if event.inaxes != self.ax1:
                return
            if event.button != 1 or event.xdata is None or event.ydata is None:
                return
            self.verts.append([event.xdata, event.ydata])
            self.poly1.set_xy(self.verts)
            self.poly2.set_xy(self.verts)
            fig.canvas.draw_idle()

        def _on_release(self, event):
            self.poly1.set_xy([])
            self.poly2.set_xy([])
            fig.canvas.draw_idle()

        def _on_select(self, verts):
            self.onselect(verts)

        def disconnect(self):
            self.lasso.disconnect_events()
            fig.canvas.mpl_disconnect(self.cid_motion)
            fig.canvas.mpl_disconnect(self.cid_press)
            fig.canvas.mpl_disconnect(self.cid_release)
            self.poly1.remove()
            self.poly2.remove()

    def onselect(verts):
        path = Path(verts)
        ind = np.nonzero(path.contains_points(data_pts))[0]
        selected_indices.clear()
        selected_indices.update(ind)

        # Highlight
        for plot in [highlight_plot["plot"], highlight_plot2["plot"]]:
            if plot:
                plot.remove()
        highlight_plot["plot"] = ax1.scatter(df.iloc[list(ind)][x_col], df.iloc[list(ind)][y_col],
                                             facecolors='none', edgecolors='red', s=40)
        highlight_plot2["plot"] = ax2.scatter(df.iloc[list(ind)][x_col], df.iloc[list(ind)][y_col],
                                              facecolors='none', edgecolors='red', s=40)

        label_text.value = ""
        label_text.layout.display = "block"
        label_text.focus()

    def on_commit(_):
        if not selected_indices:
            return
        label = label_text.value.strip() or f"Group {selection_counter['count'] + 1}"
        selection_counter["count"] += 1
        group_id = (selection_counter["count"] - 1) % (len(color_list) - 1)

        df.loc[list(selected_indices), id_column] = group_id
        df.loc[list(selected_indices), label_column] = label
        labels_dict[group_id] = label
        selected_indices.clear()

        # Clear highlight
        for key in [highlight_plot, highlight_plot2]:
            if key["plot"]:
                key["plot"].remove()
                key["plot"] = None

        label_text.layout.display = "none"
        refresh_plot()

    def on_finish(_):
        if selector_ref["selector"]:
            selector_ref["selector"].disconnect()
        plt.close(fig)
        with output:
            clear_output(wait=True)

    def on_dropdown_change(change):
        if change["type"] == "change" and change["name"] == "value":
            col = change["new"]
            if col in df.columns:
                ax2.clear()
                ax2.set_title(f"Colored by: {col}")
                vmin, vmax = np.quantile(df[col], 0.01), np.quantile(df[col], 0.99)
                ax2.scatter(df[x_col], df[y_col], s=8, c=df[col], cmap='seismic', vmin=vmin, vmax=vmax)
                fig.canvas.draw_idle()

    dropdown.observe(on_dropdown_change)

    selector_ref["selector"] = DualLasso(ax1, ax2, onselect)
    commit_button.on_click(on_commit)
    finish_button.on_click(on_finish)
    refresh_plot()

    return df



In [ ]:


for DB in DBs:

    globals()[DB]['Samp']=DB
    print(DB,globals()[DB].shape[0])

In [ ]:
%matplotlib inline

In [ ]:
AD=ad.AnnData(MCF7[NamesAll],obs=MCF7[['Samp']])

In [ ]:
AD.obsm['X_umap']=X_2d

In [ ]:
sc.pl.umap(AD,color=MRK,cmap='seismic',vmin='p1', vmax='p99',ncols=5)

In [ ]:
marker_sets = {
    # Luminal/epithelial program: epithelial & ER axis up; mesenchymal/basal down
    "Epithelial_Luminal": {
        "up":   {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    # # Basal-like program: basal keratin/mesenchymal up; luminal/epithelial down
    # "Basal_like": {
    #     "up":   {"KRT5",  "Vimentin"},
    #     "down": {"EpCAM", "E-cadherin", "ER", "GATA3", "Pan-KRT", "KRT8-18"}
    # },

    # “Basal_Noa” (histone-flavor): active H3K4 marks up; repressive H3K9me2 down
    "Basal_Noa": {
        "up":   {"H3K4me1", "H3K4me3","H3K9me2"},
        "down": {"H4K20me3","H3K36me3"}
    },

    # # EMT programs: Vimentin/aSMA/CD44 up; E-cadherin down
    # "EMT": {
    #     "up":   {"Vimentin", "aSMA", "CD44"},
    #     "down": {"E-cadherin"}
    # },
    

    # # Proliferation / cell-cycle & immediate-early signaling up
    # "Proliferation": {
    #     "up":   {"KI67", "H3S28p", "H3K9ac", "H3K64ac"},
    #     "down": set()  
    # },
}


marker_sets_extra = {
    # ---- PAM50-ish refinements ----
    "LuminalA_like": {
        "up":   {"ER", "GATA3", "KRT8-18", "E-cadherin", "EpCAM", "Pan-KRT"},
        "down": {"KRT5", "Vimentin", "aSMA", "CD44", "KI67"}  # lower proliferation bias
    },
    "LuminalB_like": {
        "up":   {"ER", "GATA3", "KRT8-18", "E-cadherin", "EpCAM", "Pan-KRT", "KI67"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },

    # ---- Hierarchy / progenitors ----
    "Luminal_Progenitor": {  # LP: EpCAM+, CD49f+, luminal keratins
        "up":   {"EpCAM", "CD49f", "KRT8-18", "Pan-KRT"},
        "down": {"KRT5", "Vimentin", "aSMA"}
    },
    "Basal_Progenitor": {    # basal/myo-biased progenitor
        "up":   {"CD49f", "KRT5", "CD44"},
        "down": {"EpCAM", "KRT8-18", "E-cadherin"}
    },
    "Myoepithelial_like": {
        "up":   {"aSMA", "KRT5", "CD49f", "Vimentin"},
        "down": {"KRT8-18", "EpCAM", "E-cadherin", "ER"}
    },

    # ---- Stemness / plasticity ----
    "CSC_CD44hi_CD24lo": {
        "up":   {"CD44", "CD49f", "BMI1", "EZH2"},
        "down": {"CD24", "E-cadherin", "ER", "GATA3"}
    },
    "Partial_EMT": {  # pEMT: epithelial retained with mesenchymal gain
        "up":   {"Vimentin", "CD44"},
        "down": set()  # keep neutral on E-cadherin here; use EMT set for strict E-cad↓
    },

    # ---- Proliferation & stress ----
    "High_Proliferation": {
        "up":   {"KI67", "H3S28p"},
        "down": set()
    },
    "DNA_Damage_Stress": {
        "up":   {"pH2A.X", "H3S28p"},
        "down": set()
    },

    # ---- Chromatin programs ----
    "Active_Chromatin": {
        "up":   {"H3K27ac", "H3K4me3", "H3K9ac", "H3K64ac", "H4K16ac", "H3K4me1"},
        "down": {"H3K27me3", "H3K9me3", "H4K20me3", "H3K27me2", "H3K9me2"}
    },
    "PRC_Repression": {  # PRC1/2 axis
        "up":   {"H3K27me3", "H2AK119ub", "EZH2", "BMI1"},
        "down": {"H3K27ac", "H3K4me3", "H3K9ac", "H4K16ac"}
    },
    "Active_Enhancer": {
        "up":   {"H3K4me1", "H3K27ac"},
        "down": {"H3K27me3"}
    },
    "Poised_Enhancer": {
        "up":   {"H3K4me1", "H3K27me3"},
        "down": {"H3K27ac"}
    },
    "Transcription_Elongation": {
        "up":   {"H3K36me3", "H3K36me2"},
        "down": set()
    },

    # ---- Epithelial integrity / adhesion ----
    "Epithelial_Adhesion": {
        "up":   {"E-cadherin", "EpCAM", "Pan-KRT", "KRT8-18"},
        "down": {"Vimentin", "aSMA", "KRT5"}
    },


}


In [ ]:
MRK=NamesAll.copy()
MRK.remove('H3')
MRK.remove('H3.3')
MRK.remove('H4')

In [ ]:
from icecream import ic
PLT={'Cycling':'#3f78c1','Basal-like':'#fb9a99','Basal':'#fb9a99','Luminal':'#33a02c','M':'gray','DNA Damage':'gray','G0':'black'}

In [ ]:
DBs

In [ ]:
for DB in DBs[:]:
#    ic(print(DB))
    CAll=globals()[f"{DB}"].copy()
    UM=umap.UMAP(min_dist=0.001,n_neighbors=50,random_state=None,verbose=True)
    X_2d=UM.fit_transform(CAll[EPC])
    AD=ad.AnnData(CAll[MRK],obs=CAll[['Samp']])
    AD.obsm['X_umap']=X_2d
#    AD=AD[AD.obs['Class']!='']
    sc.pp.sample(AD,0.25)
    Z, P, Zabs, Pabs, Zdir = sipsic_like_scores_v3(
        AD, marker_sets,
        n_perm=2048,                   # or even 16 for smoke test
        prefer_permutation=True,     # <- important
        perm_batch=512,              # keeps RAM flat
        normalize_set_weights="l2",
        use_sparse_W=False,
        progress=True               # progress bars can add overhead in some envs
    )
    
    
    sdf=pd.DataFrame(gaussian_smooth_all_torch(Z.values,AD.obsm['X_umap'],-1),columns=Z.columns)
    
    ADS=ad.AnnData(obs=sdf)
    ADS.obsm['X_umap']=AD.obsm['X_umap']
#    ADS.obs['Class']=AD.obs['Class'].values
    sc.pl.umap(AD,color=['GATA3','KRT5','H3K9me2','H3K4me1','H3K4me3'],cmap='seismic',show=False,vmin='p1',vmax='p99')
    plt.savefig(f"Plots/{DB}_UMAP.pdf",dpi=200,bbox_inches='tight')
    sc.pl.umap(ADS,color=list(ADS.obs.columns),cmap='magma_r',show=False,vcenter=0,palette=PLT)
    plt.savefig(f"Plots/{DB}_UMAP_PermCell_with_H4K20me3.pdf",dpi=200,bbox_inches='tight')
 
    ADUS=ad.AnnData(obs=Z)
    ADUS.obsm['X_umap']=AD.obsm['X_umap']
#    ADUS.obs['Class']=AD.obs['Class'].values
    sc.pl.umap(ADUS,color=list(ADUS.obs.columns),cmap='magma_r',show=False,vcenter=0,palette=PLT)
    plt.savefig(f"Plots/{DB}_UMAP_PermCell_Unsmoothed_with_H4K20me3.pdf",dpi=200,bbox_inches='tight')
    
    
    plt.show()



In [ ]:
sc.pl.umap(ADUS,color='Epithelial_Luminal',cmap='magma_r',show=False,vcenter=0,palette=PLT,vmax=1.5)
plt.savefig(f"Plots/{DB}_UMAP_PermCell_Unsmoothed_with_H4K20me3_1.pdf",dpi=200,bbox_inches='tight')


In [ ]:
sc.pl.umap(ADUS,color='Basal_Noa',cmap='magma_r',show=False,vcenter=0,palette=PLT,)
plt.savefig(f"Plots/{DB}_UMAP_PermCell_Unsmoothed_with_H4K20me3_2.pdf",dpi=200,bbox_inches='tight')


In [ ]:
X_2d=ADUS.obsm['X_umap']
pd.DataFrame(X_2d,columns=['umap1','umap2']).to_parquet("UMAP_MCF7.parquet")
#ADUS.obs

In [ ]:
ADUS.obs.to_parquet("MCF7_Sigs.parquet")

In [ ]:
sc.pl.umap(AD,color=['GATA3','KRT5','H3K9me2','H3K4me1','H3K4me3','H3K36me3'],cmap='seismic',show=False,vmin='p1',vmax='p99')
plt.savefig(f"Plots/{DB}_UMAP.pdf",dpi=200,bbox_inches='tight')


In [ ]:
AD2=AD.copy()

In [ ]:
sc.pp.subsample(AD2,0.2)

In [ ]:
AD2

In [ ]:
sc.pl.umap(AD2,color=MRK,cmap='seismic',show=False,vmin='p1',vmax='p99');
plt.savefig(f"Plots/{DB}_UMAP2.pdf",dpi=200,bbox_inches='tight')

In [ ]:
from sklearn.cluster import DBSCAN


In [ ]:
lbl=DBSCAN(eps=0.05).fit_predict(AD.obsm['X_umap'])

In [ ]:
sc.pp.neighbors(AD)


In [ ]:
sc.tl.leiden(AD,flavor="igraph" , n_iterations=2,resolution=0.15)

In [ ]:
AD.obs['CL']=lbl
AD.obs['CL']=AD.obs['CL'].astype('category')

In [ ]:
sc.pl.umap(AD,color='leiden',show=False)
plt.savefig("Plots/Projections/MCF7_Yael_CL.png",dpi=200,bbox_inches='tight')

In [ ]:
M_Cl2=AD.obs['leiden']=='2'

In [ ]:
np.savetxt("Plots/Projections/MCF7_Yael_BasalCL.csv",list(AD[M_Cl2].obs_names),fmt='%s')

In [ ]:
IDX=pd.read_csv("/Users/ronguy/Dropbox/CyTOF_Breast/data_revision_CR7_Nature/MCF7 Ori_Cl2.csv",header=None)

In [ ]:
IDX=[str(f) for f in IDX[0].values]

In [ ]:
IDX[0]

In [ ]:
AD.obs

In [ ]:
M=AD.obs_names.isin(IDX)

In [ ]:
M.sum()

In [ ]:
AD.obs['BASAL']=0
AD.obs.loc[M,'BASAL']=1

In [ ]:
AD.obs['BASAL']=AD.obs['BASAL'].astype('category')

In [ ]:
plt.scatter(AD.obsm['X_umap'][:,0],AD.obsm['X_umap'][:,1],s=.1,color='gray')
plt.scatter(AD[M].obsm['X_umap'][:,0],AD[M].obsm['X_umap'][:,1],s=1,color='red')

In [ ]:
ax=sc.pl.umap(AD,show=False)
sc.pl.umap(AD[M],color='BASAL',ax=ax,s=10,palette={1:PLT['Basal-like']},show=False)
plt.savefig("Plots/BASAL_Project.png",dpi=200,bbox_inches='tight')

In [ ]:
PLT